In [6]:
# trade_charts_daily.py
# -*- coding: utf-8 -*-
"""
Her ticker için son 1 ayın GÜNLÜK fiyatlarını yfinance'tan çeker,
ALIŞ/SATIŞ noktalarını grafiğe koyar ve sadece İŞLEM FİYATINI (adet yok) yazdırır.
PDF ÜRETMEZ. Her ticker için tek bir PNG dosyası kaydeder.

Kurulum:
    pip install pandas numpy matplotlib yfinance
Çalıştırma:
    python trade_charts_daily.py
"""

import os
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- Gömülü işlemler (senin son paylaştığın tablo) ----
# Kolonlar: Ticker, Tarih, Tutar($), Ücret, Fiyat, İşlem
TRADES = [
    ("SPGI",  "8.10.2025",   480.61,     0.96,  480.61, "ALIŞ"),
    ("AMT",   "13.10.2025",  369.30,     0.73,  184.65, "ALIŞ"),
    ("BSX",   "13.10.2025",  284.73,     0.56,   94.91, "ALIŞ"),
    ("HD",    "13.10.2025",  378.70,     0.76,  378.70, "ALIŞ"),
    ("PG",    "13.10.2025",  297.90,     0.59,  148.95, "ALIŞ"),
    ("DAL",   "15.10.2025",  122.96,     0.50,   61.48, "ALIŞ"),
    ("DAL",   "20.10.2025",  122.34,     0.50,   61.17, "SATIŞ"),
    ("NVDA",  "15.10.2025",  180.00,     1.50,  179.57, "ALIŞ"),
    ("BRK.B", "15.10.2025",  125.00,     1.50,  492.89, "ALIŞ"),
    ("AAPL",  "16.10.2025",  150.00,     1.50,  247.39, "ALIŞ"),
    ("AAPL",  "20.10.2025",  156.8090869,1.50,  258.62, "SATIŞ"),
    ("STX",   "22.10.2025",  213.13,     1.50,  213.13, "ALIŞ"),
    ("STX",   "23.10.2025",  225.03,     1.50,  225.03, "SATIŞ"),
    ("META",  "24.10.2025",  366.48,     1.50,  732.95, "ALIŞ"),
    ("VOO",   "24.10.2025",  311.29,     1.50,  622.57, "ALIŞ"),
]

# Yahoo Finance sembol eşleştirmesi (örn. BRK.B -> BRK-B)
YF_MAPPING = {
    "BRK.B": "BRK-B",
}

def parse_date_tr(x: str) -> datetime:
    for fmt in ("%d.%m.%Y", "%d.%m.%y", "%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(str(x), fmt)
        except Exception:
            pass
    # yine de parse edilemezse NaT dönebilir
    return pd.to_datetime(x, dayfirst=True, errors="coerce")

def prepare_trades(trades):
    df = pd.DataFrame(trades, columns=["Ticker","Tarih","Amount","Fee","TradePrice","Side"])
    df["Date"] = df["Tarih"].apply(parse_date_tr)
    # sadece ihtiyacımız olan kolonlar
    df = df[["Ticker","Date","TradePrice","Side"]].sort_values(["Ticker","Date"]).reset_index(drop=True)
    return df

def fetch_daily_prices(ticker: str, start: datetime, end: datetime) -> pd.Series:
    """yfinance'tan günlük kapanışlar. Başarısız olursa boş döner."""
    try:
        import yfinance as yf
        yf_ticker = YF_MAPPING.get(ticker, ticker)
        data = yf.download(yf_ticker, start=start, end=end, interval="1d", progress=False, threads=True)
        if data is None or len(data) == 0:
            return pd.Series(dtype=float)
        close = data["Close"].copy()
        close.name = "Close"
        return close
    except Exception:
        return pd.Series(dtype=float)

def plot_ticker_daily(ax, close: pd.Series, trades_tkr: pd.DataFrame, ticker: str):
    # Fiyat çizgisi
    ax.plot(close.index, close.values, linewidth=1.5)
    ax.set_title(f"{ticker} — Son 1 Ay Günlük Fiyat")
    ax.set_xlabel("Tarih"); ax.set_ylabel("Fiyat")
    ax.grid(True, linestyle="--", alpha=0.3)

    # ALIŞ / SATIŞ noktaları (sadece fiyat etiketi)
    buys  = trades_tkr[trades_tkr["Side"].str.upper().str.startswith("ALI")]
    sells = trades_tkr[~trades_tkr["Side"].str.upper().str.startswith("ALI")]

    # ALIŞ: ▲ ve fiyat etiketi
    if not buys.empty:
        ax.scatter(buys["Date"], buys["TradePrice"], marker="^", s=70, label="ALIŞ")
        for _, r in buys.iterrows():
            ax.annotate(f"{r['TradePrice']:.2f}", (r["Date"], r["TradePrice"]),
                        textcoords="offset points", xytext=(0,8), ha="center", fontsize=8)

    # SATIŞ: ▼ ve fiyat etiketi
    if not sells.empty:
        ax.scatter(sells["Date"], sells["TradePrice"], marker="v", s=70, label="SATIŞ")
        for _, r in sells.iterrows():
            ax.annotate(f"{r['TradePrice']:.2f}", (r["Date"], r["TradePrice"]),
                        textcoords="offset points", xytext=(0,-12), ha="center", fontsize=8)

    if (not buys.empty) or (not sells.empty):
        ax.legend(loc="best")

def main():
    os.makedirs("charts", exist_ok=True)

    trades = prepare_trades(TRADES)
    tickers = trades["Ticker"].unique()

    # Pencere: son işlemin tarihine göre 1 ay (yoksa bugüne göre)
    max_trade_dt = trades["Date"].max()
    base_end = max_trade_dt if pd.notna(max_trade_dt) else datetime.utcnow()
    start = (base_end - timedelta(days=31)).replace(hour=0, minute=0, second=0, microsecond=0)
    end   = (base_end + timedelta(days=1)).replace(hour=0, minute=0, second=0, microsecond=0)

    for tkr in tickers:
        tdf = trades[trades["Ticker"] == tkr].copy()

        # Günlük kapanışları çek
        close = fetch_daily_prices(tkr, start, end)

        # Eğer veri gelmezse, en azından trade noktalarını çizmek için bir fallback çizgi yapalım:
        # (kullanıcı "gün gün verileri çek" dedi, o yüzden fallback kullanmamak daha doğru;
        #  yine de boş kalmasın diye basit trade-bazlı çizgi ekliyoruz)
        if close.empty:
            # Trade günleri için seri, diğer günler NaN; ffill ile basit çizgi
            idx = pd.date_range(start=start, end=end, freq="D")
            s = pd.Series(index=pd.to_datetime(tdf["Date"]), data=tdf["TradePrice"].values).sort_index()
            s = s[~s.index.duplicated(keep="last")]
            close = s.reindex(idx).ffill()
            close.name = "Close"

        # Çiz ve kaydet
        fig, ax = plt.subplots(figsize=(10, 5))
        plot_ticker_daily(ax, close, tdf, tkr)
        plt.tight_layout()
        out_png = os.path.join("charts", f"{tkr}.png")
        fig.savefig(out_png, dpi=150)
        plt.close(fig)

    print("Tamamdır. Grafikler charts/ klasörüne kaydedildi (her ticker için 1 PNG).")

if __name__ == "__main__":
    main()


Tamamdır. Grafikler charts/ klasörüne kaydedildi (her ticker için 1 PNG).


In [4]:
import pandas as pd
from pathlib import Path

# --- Ayarlar ---
EXCEL_PATH = Path("1-) Investors.xlsx")  # Gerekirse tam yol ver: Path(r"C:\...\Investors.xlsx")
SHEET_CANDIDATES = ["İşlem Geçmişi", "Islem Gecmisi", "İslem Gecmisi", "Islem Geçmişi"]

def _tr_to_float(x):
    """
    '480,61' -> 480.61
    Boş/NaN/None -> NaN
    """
    if pd.isna(x):
        return pd.NA
    if isinstance(x, (int, float)):
        return x
    s = str(x).strip()
    # Binlik ayıracı yoksa direkt virgülü noktaya çevir
    s = s.replace(".", "").replace(",", ".") if s.count(",") == 1 and s.count(".") <= 1 else s.replace(",", ".")
    try:
        return float(s)
    except Exception:
        return pd.NA

def load_islem_gecmisi(path: Path = EXCEL_PATH) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dosya bulunamadı: {path.resolve()}")

    # Sheet adını esnek bul
    xl = pd.ExcelFile(path)
    sheet_name = None
    # Doğrudan eşleşme
    for cand in SHEET_CANDIDATES:
        if cand in xl.sheet_names:
            sheet_name = cand
            break
    # Harf-normalize ile yaklaşık eşleşme
    if sheet_name is None:
        norm = lambda s: (s
                          .replace("İ","I").replace("ı","i")
                          .replace("Ş","S").replace("ş","s")
                          .replace("Ğ","G").replace("ğ","g")
                          .replace("Ü","U").replace("ü","u")
                          .replace("Ö","O").replace("ö","o"))
        target = norm("Islem Gecmisi")
        for s in xl.sheet_names:
            if norm(s).lower() == target.lower():
                sheet_name = s
                break

    if sheet_name is None:
        raise ValueError(f"Sheet bulunamadı. Mevcut sheet'ler: {xl.sheet_names}")

    # Oku


# İşlem Geçmişinden Açık Pozisyon ve PnL e geçiş

In [14]:
import pandas as pd
from pathlib import Path

# === Ayarlar ===
EXCEL_PATH = Path("1-) Investors.xlsx")   # Dosya adını AYNEN kullan
SHEET_TARGET = "İşlem Geçmişi"

def _tr_to_float(x):
    if pd.isna(x):
        return pd.NA
    if isinstance(x, (int, float)):
        return float(x)
    s = str(x).strip()
    # "480,61" -> 480.61; binlik ayıraçları da temizle
    s = s.replace(".", "").replace(",", ".")
    try:
        return float(s)
    except Exception:
        return pd.NA

def load_islem_gecmisi(path: Path = EXCEL_PATH, sheet_name: str = SHEET_TARGET) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Dosya bulunamadı: {path.resolve()}")

    # Sheet adını net biliyorsak direkt oku; değilse yedek eşleştirme yap
    try:
        df = pd.read_excel(path, sheet_name=sheet_name, engine="openpyxl")
    except ValueError:
        # Yaklaşık eşleşme (İ/ı/ş/ğ/ü/ö farklarını tolere et)
        xl = pd.ExcelFile(path)
        def norm(s):
            return (s.replace("İ","I").replace("ı","i")
                     .replace("Ş","S").replace("ş","s")
                     .replace("Ğ","G").replace("ğ","g")
                     .replace("Ü","U").replace("ü","u")
                     .replace("Ö","O").replace("ö","o")).lower()
        target = norm(sheet_name)
        match = None
        for s in xl.sheet_names:
            if norm(s) == target:
                match = s
                break
        if match is None:
            raise ValueError(f"Sheet bulunamadı. Mevcut: {xl.sheet_names}")
        df = pd.read_excel(path, sheet_name=match, engine="openpyxl")

    # Sık kolon adlarını normalize et
    rename_map = {
        "Islem": "İşlem",
        "Islem Ucreti": "İşlem Ücreti",
        "O anki fiyat": "O anki fiyat",
        "Islem Donusu ($)": "İşlem Dönüşü ($)",
        "Platform": "Platform",
        "Tutar ($)": "Tutar ($)",
        "Tarih": "Tarih",
        "Ticker": "Ticker",
        "Şevval": "Şevval",
        "Barış": "Barış",
        "Oğuz": "Oğuz",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    # Tarihi gün.ay.yıl olarak parse et
    if "Tarih" in df.columns:
        df["Tarih"] = pd.to_datetime(df["Tarih"], dayfirst=True, errors="coerce")

    # TR sayı formatlı kolonları çevir
    for col in ["Tutar ($)", "İşlem Ücreti", "O anki fiyat", "İşlem Dönüşü ($)", "Şevval", "Barış", "Oğuz"]:
        if col in df.columns:
            df[col] = df[col].map(_tr_to_float)

    return df

# === Kullanım ===
if __name__ == "__main__":
    df = load_islem_gecmisi()
    print(df.head())

    # Yalnızca ALIŞ işlemlerinin toplam tutarı
    if "İşlem" in df.columns and "Tutar ($)" in df.columns:
        toplam_alis = df.loc[df["İşlem"].str.upper() == "ALIŞ", "Tutar ($)"].sum(skipna=True)
        print("Toplam ALIŞ Tutarı:", toplam_alis)

    # İstersen ticker bazında ALIŞ toplamı:
    if {"İşlem", "Ticker", "Tutar ($)"}.issubset(df.columns):
        grp = (df[df["İşlem"].str.upper() == "ALIŞ"]
               .groupby("Ticker", as_index=False)["Tutar ($)"].sum()
               .rename(columns={"Tutar ($)": "Toplam ALIŞ ($)"}))
        print("\nTicker bazında ALIŞ toplamı:\n", grp)


  Ticker Name      Tarih Tutar ($) İşlem Ücreti O anki fiyat İşlem  \
0        SPGI 2025-10-08    480.61         0.96       480.61  ALIŞ   
1         AMT 2025-10-13     369.3         0.73       184.65  ALIŞ   
2         BSX 2025-10-13    284.73         0.56        94.91  ALIŞ   
3          HD 2025-10-13     378.7         0.76        378.7  ALIŞ   
4          PG 2025-10-13     297.9         0.59       148.95  ALIŞ   

  İşlem Dönüşü ($) Şevval Barış    Oğuz Platform  User ID  
0          -481.57   <NA>  <NA> -481.57       YK      1.0  
1          -370.03   <NA>  <NA> -370.03       YK      1.0  
2          -285.29   <NA>  <NA> -285.29       YK      1.0  
3          -379.46   <NA>  <NA> -379.46       YK      1.0  
4          -298.49   <NA>  <NA> -298.49       YK      1.0  
Toplam ALIŞ Tutarı: 3280.1000000000004


In [13]:
df

,Ticker Name,Tarih,Tutar ($),İşlem Ücreti,O anki fiyat,İşlem,İşlem Dönüşü ($),Şevval,Barış,Ben,Platform,User ID
0,SPGI,2025-10-08,480.61,0.96,480.61,ALIŞ,-481.57,<NA>,<NA>,-481.570000,YK,1.0
1,AMT,2025-10-13,369.3,0.73,184.65,ALIŞ,-370.03,<NA>,<NA>,-370.030000,YK,1.0
2,BSX,2025-10-13,284.73,0.56,94.91,ALIŞ,-285.29,<NA>,<NA>,-285.290000,YK,1.0
3,HD,2025-10-13,378.7,0.76,378.7,ALIŞ,-379.46,<NA>,<NA>,-379.460000,YK,1.0
4,PG,2025-10-13,297.9,0.59,148.95,ALIŞ,-298.49,<NA>,<NA>,-298.490000,YK,1.0
5,DAL,2025-10-15,122.96,0.5,61.48,ALIŞ,-123.46,<NA>,<NA>,-123.460000,YK,1.0
6,DAL,2025-10-20,122.34,0.5,61.17,SATIŞ,121.84,<NA>,<NA>,121.840000,YK,1.0
7,NVDA,2025-10-15,180.0,1.5,179.57,ALIŞ,-181.5,-60.75,-60.75,-60.000000,Midas,2.0
8,BRK.B,2025-10-15,125.0,1.5,492.89,ALIŞ,-126.5,-42.416667,-42.416667,-41.666667,Midas,2.0
9,AAPL,2025-10-16,150.0,1.5,247.39,ALIŞ,-151.5,-50.75,-50.75,-50.000000,Midas,2.0
